<a href="https://colab.research.google.com/github/kanchanr8595-create/CareFlow-Clinical-Pathway-Process-Mining/blob/development_akshaya/Transition%20result.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

transition=pd.read_csv("/content/transition_time_results (1).csv")

raw=pd.read_csv("/content/careflow_synthetic_500_patients.csv")

print(raw.head())
print(transition.head())

  case_id   event_id  event_order             activity            timestamp  \
0   P0001  P0001-E01            1         Registration  2026-09-11 09:54:00   
1   P0001  P0001-E02            2               Triage  2026-09-11 10:20:00   
2   P0001  P0001-E03            3  Doctor Consultation  2026-09-11 11:44:00   
3   P0001  P0001-E04            4             Lab Test  2026-09-11 12:05:00   
4   P0001  P0001-E05            5              Imaging  2026-09-11 13:30:00   

   department journey_type  is_loop_back data_source  
0   Reception       normal         False   synthetic  
1   Emergency       normal         False   synthetic  
2   Emergency       normal         False   synthetic  
3  Laboratory       normal         False   synthetic  
4   Radiology       normal         False   synthetic  
              activity        next_activity  transition_count  \
0              Imaging            Treatment                96   
1  Doctor Consultation            Treatment                91   


In [8]:
print("\n Missing Values:")
print(raw.isnull().sum())

print("\n Exact Duplicates:")
print(raw.duplicated().sum())

print("Dupliacte patient-activity-timestamp records:")
print(raw.duplicated(["case_id","activity","timestamp"]).sum())


 Missing Values:
case_id         0
event_id        0
event_order     0
activity        0
timestamp       0
department      0
journey_type    0
is_loop_back    0
data_source     0
dtype: int64

 Exact Duplicates:
0
Dupliacte patient-activity-timestamp records:
0


In [11]:
raw["timestamp"]=pd.to_datetime(raw["timestamp"],errors="coerce")

raw=raw.sort_values(["case_id","event_order"]).copy()

raw["next_timestamp"]=(raw.groupby("case_id")["timestamp"].shift(-1))

raw["time_gap"]=(raw["next_timestamp"]-raw["timestamp"]).dt.total_seconds()/60

print("Negative gaps:",(raw["time_gap"]<0).sum())

print("invalid timestamps:",raw["timestamp"].isna().sum())

Negative gaps: 0
invalid timestamps: 0


In [18]:
suspicious=raw[
    raw["time_gap"]>143.5
]

print(suspicious[
      ["case_id","activity","next_timestamp","time_gap","journey_type"]
])

     case_id             activity      next_timestamp  time_gap journey_type
25     P0004               Triage 2026-09-06 17:32:00     186.0      delayed
33     P0005              Imaging 2026-09-05 20:22:00     187.0      delayed
43     P0007               Triage 2026-09-03 21:23:00     232.0      delayed
45     P0007             Lab Test 2026-09-04 00:54:00     173.0      delayed
70     P0011               Triage 2026-09-03 19:16:00     231.0      delayed
...      ...                  ...                 ...       ...          ...
3158   P0470               Triage 2026-09-02 19:49:00     185.0      delayed
3159   P0470  Doctor Consultation 2026-09-02 22:39:00     170.0      delayed
3217   P0480               Triage 2026-09-11 14:37:00     180.0      delayed
3219   P0480             Lab Test 2026-09-11 19:08:00     220.0      delayed
3346   P0500               Triage 2026-09-06 14:28:00     213.0      delayed

[143 rows x 5 columns]


In [20]:
raw["next_activity"]=(
    raw.groupby("case_id")["activity"].shift(-1)
)

result=raw.groupby(
    ["activity","next_activity","journey_type"]
)["time_gap"].agg(
    ["count","mean","median","max"]
).reset_index()

print(result)

               activity        next_activity journey_type  count        mean  \
0             Discharge            Follow-up      complex     10   24.900000   
1             Discharge            Follow-up      delayed     23   20.260870   
2             Discharge            Follow-up    loop_back     16   17.625000   
3             Discharge            Follow-up       normal     48   19.833333   
4   Doctor Consultation              Imaging      delayed      7   40.857143   
5   Doctor Consultation              Imaging       normal     23   40.000000   
6   Doctor Consultation             Lab Test      complex     96   47.218750   
7   Doctor Consultation             Lab Test      delayed     74   50.162162   
8   Doctor Consultation             Lab Test    loop_back    122   47.909836   
9   Doctor Consultation             Lab Test       normal    196   51.545918   
10  Doctor Consultation            Treatment      delayed     24  165.541667   
11  Doctor Consultation            Treat

In [21]:
raw.to_csv(
    "days3_validated_patient_events.csv",
    index=False
)

transition.to_csv("transition_result.csv",index=False)